In [1]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /home/user1/WhatDoLLMsWant


# Books domain — design (2026-10-12)

A second feature domain next to the laptops. Items are **real, famous books**; the prompt shows only
**title, author and year** (no tags), so the model chooses from what it knows about the book. The
**features are hidden**: the genre shelves Goodreads readers put the book on. They are used only in
the analysis (feature BT on the tags) and in contracts ("I prefer fantasy.").

Source: [goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k) (10,000 most-rated books on
Goodreads, with per-book shelf counts), downloaded to `external/goodbooks-10k/` (gitignored).

Decisions (Lotan, 2026-10-12): real books, as famous as possible; genre tags only for now;
prompt = title + author + year.

**The list itself is built by [src/books/make_books.py](../src/books/make_books.py)** (reproducible,
writes `src/books/books.csv`, which `alternatives.books` loads). This notebook calls that script's
functions and shows the checks behind the choices.

In [2]:
import pandas as pd
from src.books import make_books as mb
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 60)
genres = mb.GENRE_LIST
books, bt = mb.load_raw()
print(len(books), "books,", bt.tag_name.nunique(), "distinct shelves")

10000 books, 34252 distinct shelves


## 1. From shelves to genre tags

Shelves are user-made, so most of the volume is bookkeeping ("to-read", "owned", "favorites") and the
same genre has many spellings ("sci-fi", "scifi", "science-fiction"). `make_books.GENRES` keeps 8 genres
and maps the common spellings onto them. `children` and `horror` were dropped: too rare among the famous
books (3 each in the top 40).

In [3]:
share = mb.genre_shares(bt)    # per book: share of its genre shelvings, rows sum to 1
share.head()

genre,classics,fantasy,historical-fiction,mystery-thriller,nonfiction,romance,science-fiction,young-adult
goodreads_book_id,,,,,,,,
1,0.015600,0.669169,0.0,0.007572,0.0,0.005439,0.000000,0.302221
2,0.029701,0.522337,0.0,0.054860,0.0,0.026141,0.024423,0.342538
3,0.030331,0.677466,0.0,0.008487,0.0,0.000000,0.000000,0.283716
5,0.017107,0.683198,0.0,0.008255,0.0,0.000000,0.000000,0.291440
6,0.021655,0.654291,0.0,0.008282,0.0,0.000000,0.000000,0.315772


A book **has** a genre if that genre is at least 12% of its genre shelvings (`THR`). The threshold
sets how many tags a book gets; 0.12 gives mostly 1–3 tags, which is what a reader would say.

In [4]:
top500 = books.sort_values("ratings_count", ascending=False).goodreads_book_id.head(500)
for thr in [0.08, 0.12, 0.20]:
    n = (share >= thr).sum(axis=1).reindex(top500)
    print(f"thr={thr}: tags per book (top 500 by ratings) ->", n.value_counts().sort_index().to_dict())
print("THR used:", mb.THR)

thr=0.08: tags per book (top 500 by ratings) -> {1: 141, 2: 194, 3: 128, 4: 35, 5: 2}
thr=0.12: tags per book (top 500 by ratings) -> {1: 188, 2: 194, 3: 110, 4: 8}
thr=0.2: tags per book (top 500 by ratings) -> {1: 259, 2: 201, 3: 40}
THR used: 0.12


In [5]:
b = mb.candidates(books, share)
print("share of the 500 most-rated books carrying each tag:")
print(b.head(500)[genres].mean().round(2).to_string())

share of the 500 most-rated books carrying each tag:
fantasy               0.32
science-fiction       0.13
romance               0.24
mystery-thriller      0.15
young-adult           0.36
classics              0.39
historical-fiction    0.14
nonfiction            0.14


## 2. Picking the items

Fame first, then balance (`make_books.pick`). Three passes, always one book per series (its first
book) and at most two per author:

1. the **32 most-rated** books, no condition on tags;
2. every tag up to **8 books**, taking the most-rated books that carry it (this is what brings in
   nonfiction and historical fiction);
3. fill up to **48** by rating count, with a cap of 18 per tag (or, if pass 2 overshot, drop the
   last-added books that no tag needs).

Why 48: 48 × 47 = 2,256 ordered pairs × 5 templates = 11,280 rows per run, a bit more than the
45 laptops (9,900). Why a minimum of 8 per tag: a tag weight fitted on fewer books is noise.

In [6]:
sel, cnt = mb.pick(b)
sel["pop_rank_all"] = [int((books.ratings_count > rc).sum()) + 1 for rc in sel.ratings_count]
sel[["title_clean", "author", "year", "average_rating", "ratings_count", "pop_rank_all", "tags"]]

,title_clean,author,year,average_rating,ratings_count,pop_rank_all,tags
goodreads_book_id,,,,,,,
2767052,The Hunger Games,Suzanne Collins,2008,4.34,4780653,1,"science-fiction, young-adult"
3,Harry Potter and the Sorcerer's Stone,J.K. Rowling,1997,4.44,4602479,2,"fantasy, young-adult"
41865,Twilight,Stephenie Meyer,2005,3.57,3866839,3,"fantasy, young-adult"
2657,To Kill a Mockingbird,Harper Lee,1960,4.25,3198671,4,classics
4671,The Great Gatsby,F. Scott Fitzgerald,1925,3.89,2683664,5,classics
11870085,The Fault in Our Stars,John Green,2012,4.26,2346404,6,"romance, young-adult"
5907,The Hobbit,J.R.R. Tolkien,1937,4.25,2071616,7,"fantasy, classics"
5107,The Catcher in the Rye,J.D. Salinger,1951,3.79,2044241,8,"young-adult, classics"
1885,Pride and Prejudice,Jane Austen,1813,4.24,2035490,9,"romance, classics"


## 3. Checks: tag coverage, tag overlap, fame

In [7]:
print("books per tag:", cnt)
print("least famous pick is #%d of 10,000 by ratings count" % sel.pop_rank_all.max())
print("tags per book:", sel.n_tags.value_counts().sort_index().to_dict())
sel[genres].corr().round(2)

books per tag: {'fantasy': 10, 'science-fiction': 8, 'romance': 8, 'mystery-thriller': 8, 'young-adult': 13, 'classics': 26, 'historical-fiction': 8, 'nonfiction': 8}
least famous pick is #119 of 10,000 by ratings count
tags per book: {1: 16, 2: 23, 3: 9}


,fantasy,science-fiction,romance,mystery-thriller,young-adult,classics,historical-fiction,nonfiction
fantasy,1.00,-0.09,-0.23,0.05,0.26,-0.04,-0.23,-0.09
science-fiction,-0.09,1.00,-0.20,-0.20,0.10,0.07,-0.20,-0.20
romance,-0.23,-0.20,1.00,-0.20,-0.15,-0.04,0.25,-0.05
mystery-thriller,0.05,-0.20,-0.20,1.00,-0.02,-0.26,-0.20,-0.20
young-adult,0.26,0.10,-0.15,-0.02,1.00,-0.19,-0.02,-0.27
classics,-0.04,0.07,-0.04,-0.26,-0.19,1.00,0.07,-0.15
historical-fiction,-0.23,-0.20,0.25,-0.20,-0.02,0.07,1.00,-0.20
nonfiction,-0.09,-0.20,-0.05,-0.20,-0.27,-0.15,-0.20,1.00


Correlations stay under 0.4 in absolute value (fantasy–young-adult is the largest, +0.31), so the
8 tag weights are separately identifiable. `classics` is on 26 of the 48 books: famous books are old
books, there is no way around it with "as famous as possible".

## 4. The committed list and the prompt

`src/books/books.csv` is what the collector uses. Check it matches this run, then show the exact
prompt the model sees (frame + contract + template + items), built by the real collector code.

In [8]:
committed = pd.read_csv("src/books/books.csv")
assert list(committed.goodreads_book_id) == list(sel.index), "src/books/books.csv is stale - run src/books/make_books.py"
print("src/books/books.csv matches:", len(committed), "books")

src/books/books.csv matches: 48 books


In [9]:
import importlib.util, sys, types
sys.modules.setdefault("src.agent", types.SimpleNamespace(agent_factory=None))   # the collector imports torch through src.agent; not needed here
spec = importlib.util.spec_from_file_location("dc", "scripts/data_collection.py")
dc = importlib.util.module_from_spec(spec); spec.loader.exec_module(dc)
import src.prompts, src.alternatives
items = src.alternatives.books
tmpl = src.prompts.options_comparisons[0]
for con in [{}, {"fantasy": "yes"}, {"fantasy": "yes", "classics": "yes"}]:
    prefix = " ".join(p for p in (dc.FRAMES["reading"], dc.constraints_text(con)) if p)
    print("---- constraints:", con)
    print(f"{prefix}\n{tmpl}".format(A=dc.item_text(items[6]), B=dc.item_text(items[11])))
print("---- base-model format, no contract:")
print(src.prompts.pretrained_options_comparisons[0].format(A=dc.item_text(items[6]), B=dc.item_text(items[11])))

---- constraints: {}
I am looking for a book to read.
You have two options:
Option 1: The Hobbit by J.R.R. Tolkien (1937)
Option 2: 1984 by George Orwell (1949)
Which do you prefer?
Answer: 
---- constraints: {'fantasy': 'yes'}
I am looking for a book to read. I prefer fantasy.
You have two options:
Option 1: The Hobbit by J.R.R. Tolkien (1937)
Option 2: 1984 by George Orwell (1949)
Which do you prefer?
Answer: 
---- constraints: {'fantasy': 'yes', 'classics': 'yes'}
I am looking for a book to read. I prefer fantasy and classics.
You have two options:
Option 1: The Hobbit by J.R.R. Tolkien (1937)
Option 2: 1984 by George Orwell (1949)
Which do you prefer?
Answer: 
---- base-model format, no contract:
Option 1: The Hobbit by J.R.R. Tolkien (1937)
Option 2: 1984 by George Orwell (1949)
My preference between these two is Option 
